Imports  PySpark functions

In [0]:
from pyspark.sql import functions as F

In [0]:
from pyspark.sql.functions import (
    col,
    when,
    round
)

Load Silver tables

In [0]:
silver_fuzzy = spark.table("workspace.default.silver_fuzzy_matching")
silver_marketplace = spark.table("workspace.default.silver_marketplace")
silver_products = spark.table("workspace.default.silver_products")
silver_bom = spark.table("workspace.default.silver_bom")
silver_warranty = spark.table("workspace.default.silver_warranty")

Create gold base

In [0]:
display(
    silver_fuzzy
    .join(
        silver_marketplace,
        silver_fuzzy["listing_id"] == silver_marketplace["listing_id"],
        "left"
    )
    .join(
        silver_products,
        silver_fuzzy["matched_sku"] == silver_products["sku"],
        "left"
    )
    .select(
        silver_fuzzy["listing_id"],
        silver_fuzzy["matched_sku"],
        silver_marketplace["price"],
        silver_products["original_price"],
        silver_products["product_name"]
    )
    .orderBy("listing_id")
)

listing_id,matched_sku,price,original_price,product_name
M001,LAP1001,42000.0,75000,EcoBook Pro 14
M002,LAP1001,38000.0,75000,EcoBook Pro 14
M003,LAP1001,9000.0,75000,EcoBook Pro 14
M004,LAP1001,4000.0,75000,EcoBook Pro 14
M005,LAP1001,2200.0,75000,EcoBook Pro 14
M006,LAP1002,36000.0,68000,EcoBook Air 13
M007,LAP1002,32000.0,68000,EcoBook Air 13
M008,LAP1002,8200.0,68000,EcoBook Air 13
M009,LAP1003,26000.0,52000,EcoBook Basic 15
M010,LAP1003,23000.0,52000,EcoBook Basic 15


In [0]:
gold_base = (
    silver_fuzzy.alias("f")
    .join(
        silver_marketplace.alias("m"),
        col("f.listing_id") == col("m.listing_id"),
        "left"
    )
    .join(
        silver_products.alias("p"),
        col("f.matched_sku") == col("p.sku"),
        "left"
    )
    .select(
        col("f.listing_id").alias("listing_id"),
        col("f.marketplace_title").alias("marketplace_title"),
        col("f.matched_sku").alias("sku"),
        col("f.matched_product").alias("product_name"),
        col("m.listing_date"),
        col("m.listing_type"),
        col("m.price").alias("secondary_market_price"),
        col("m.condition"),
        col("p.category"),
        col("p.launch_year"),
        col("p.original_price"),
        col("f.distance"),
        col("f.token_score"),
        col("f.size_match"),
        col("f.weighted_score")
    )
)

In [0]:
display(
    gold_base
    .orderBy("listing_id")
)

listing_id,marketplace_title,sku,product_name,listing_date,listing_type,secondary_market_price,condition,category,launch_year,original_price,distance,token_score,size_match,weighted_score
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,EcoBook Pro 14,2026-07-01,product,42000.0,Good,Laptop,2024,75000,12,0.6,1,0.5
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,EcoBook Pro 14,2026-07-03,product,38000.0,Used,Laptop,2024,75000,16,0.42857142857142855,1,0.41428571428571426
M003,Eco Book Pro 14 Screen Display,LAP1001,EcoBook Pro 14,2026-07-05,component,9000.0,Used,Laptop,2024,75000,16,0.2857142857142857,1,0.34285714285714286
M004,EcoBook Pro 14 Battery Original,LAP1001,EcoBook Pro 14,2026-07-06,component,4000.0,Used,Laptop,2024,75000,17,0.6,1,0.5
M005,EcoBook Pro 14 Motherboard,LAP1001,EcoBook Pro 14,2026-07-07,component,2200.0,For Parts,Laptop,2024,75000,12,0.75,1,0.575
M006,EcoBook Air 13 Laptop,LAP1002,EcoBook Air 13,2026-07-02,product,36000.0,Good,Laptop,2023,68000,7,0.75,1,0.575
M007,EcoBook Air13 13-inch Used,LAP1002,EcoBook Air 13,2026-07-04,product,32000.0,Fair,Laptop,2023,68000,13,0.6,1,0.5
M008,EcoBook Air 13 Display Panel,LAP1002,EcoBook Air 13,2026-07-08,component,8200.0,Used,Laptop,2023,68000,14,0.6,1,0.5
M009,EcoBook Basic 15 Laptop,LAP1003,EcoBook Basic 15,2026-07-02,product,26000.0,Good,Laptop,2022,52000,7,0.75,1,0.575
M010,EcoBook Basic15 Used Notebook,LAP1003,EcoBook Basic 15,2026-07-09,product,23000.0,Used,Laptop,2022,52000,14,0.6,1,0.5


Calculate secondary
market depreciation -Calculates how much value a product has lost between its original 
price and its secondary-market price. 

In [0]:
gold_base = gold_base.withColumn(
    "secondary_market_depreciation",
    when(
        col("original_price") > 0,
        (
            (col("original_price") - col("secondary_market_price"))
            / col("original_price")
        ) * 100
    ).otherwise(0.0)
)

In [0]:
display(
    gold_base.select(
        "listing_id",
        "sku",
        "original_price",
        "secondary_market_price",
        "secondary_market_depreciation"
    ).orderBy("listing_id")
)

listing_id,sku,original_price,secondary_market_price,secondary_market_depreciation
M001,LAP1001,75000,42000.0,44.0
M002,LAP1001,75000,38000.0,49.333333333333336
M003,LAP1001,75000,9000.0,88.0
M004,LAP1001,75000,4000.0,94.66666666666667
M005,LAP1001,75000,2200.0,97.06666666666666
M006,LAP1002,68000,36000.0,47.05882352941176
M007,LAP1002,68000,32000.0,52.94117647058824
M008,LAP1002,68000,8200.0,87.94117647058823
M009,LAP1003,52000,26000.0,50.0
M010,LAP1003,52000,23000.0,55.769230769230774


In [0]:
display(
    silver_bom.orderBy("sku", "component_id")
)

sku,component_id,component_name,component_cost
LAP1001,C001,Motherboard,18000.0
LAP1001,C002,Display,12000.0
LAP1001,C003,Battery,6000.0
LAP1001,C004,SSD,5000.0
LAP1001,C005,Keyboard,2500.0
LAP1001,C006,Cooling Fan,1800.0
LAP1002,C001,Motherboard,18900.0
LAP1002,C002,Display,12600.0
LAP1002,C003,Battery,6300.0
LAP1002,C004,SSD,5250.0


In [0]:
display(
    silver_warranty.orderBy("sku", "component_id")
)

sku,component_id,units_sold,failure_count,failure_rate
LAP1001,C001,750,206,0.2747
LAP1001,C002,750,29,0.0387
LAP1001,C003,750,132,0.176
LAP1001,C004,750,51,0.068
LAP1001,C005,750,44,0.0587
LAP1001,C006,750,66,0.088
LAP1002,C001,800,228,0.285
LAP1002,C002,800,33,0.0412
LAP1002,C003,800,147,0.1838
LAP1002,C004,800,57,0.0712


In [0]:
silver_bom = spark.table("workspace.default.silver_bom")
silver_warranty = spark.table("workspace.default.silver_warranty")
silver_fuzzy = spark.table("workspace.default.silver_fuzzy_matching")
silver_marketplace = spark.table("workspace.default.silver_marketplace")
silver_products = spark.table("workspace.default.silver_products")

Create 
component_metrics-Joins BOM and Warranty using SKU and Component ID so that 
component cost and component failure information are available 
together.

In [0]:
component_metrics = (
    silver_bom.alias("b")
    .join(
        silver_warranty.alias("w"),
        (col("b.sku") == col("w.sku")) &
        (col("b.component_id") == col("w.component_id")),
        "left"
    )
    .select(
        col("b.sku").alias("sku"),
        col("b.component_id").alias("component_id"),
        col("b.component_name").alias("component_name"),
        col("b.component_cost").alias("component_cost"),
        col("w.units_sold").alias("units_sold"),
        col("w.failure_count").alias("failure_count"),
        col("w.failure_rate").alias("failure_rate")
    )
)

In [0]:
display(gold_base.orderBy("listing_id"))

listing_id,marketplace_title,sku,product_name,listing_date,listing_type,secondary_market_price,condition,category,launch_year,original_price,distance,token_score,size_match,weighted_score,secondary_market_depreciation
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,EcoBook Pro 14,2026-07-01,product,42000.0,Good,Laptop,2024,75000,12,0.6,1,0.5,44.0
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,EcoBook Pro 14,2026-07-03,product,38000.0,Used,Laptop,2024,75000,16,0.42857142857142855,1,0.41428571428571426,49.333333333333336
M003,Eco Book Pro 14 Screen Display,LAP1001,EcoBook Pro 14,2026-07-05,component,9000.0,Used,Laptop,2024,75000,16,0.2857142857142857,1,0.34285714285714286,88.0
M004,EcoBook Pro 14 Battery Original,LAP1001,EcoBook Pro 14,2026-07-06,component,4000.0,Used,Laptop,2024,75000,17,0.6,1,0.5,94.66666666666667
M005,EcoBook Pro 14 Motherboard,LAP1001,EcoBook Pro 14,2026-07-07,component,2200.0,For Parts,Laptop,2024,75000,12,0.75,1,0.575,97.06666666666666
M006,EcoBook Air 13 Laptop,LAP1002,EcoBook Air 13,2026-07-02,product,36000.0,Good,Laptop,2023,68000,7,0.75,1,0.575,47.05882352941176
M007,EcoBook Air13 13-inch Used,LAP1002,EcoBook Air 13,2026-07-04,product,32000.0,Fair,Laptop,2023,68000,13,0.6,1,0.5,52.94117647058824
M008,EcoBook Air 13 Display Panel,LAP1002,EcoBook Air 13,2026-07-08,component,8200.0,Used,Laptop,2023,68000,14,0.6,1,0.5,87.94117647058823
M009,EcoBook Basic 15 Laptop,LAP1003,EcoBook Basic 15,2026-07-02,product,26000.0,Good,Laptop,2022,52000,7,0.75,1,0.575,50.0
M010,EcoBook Basic15 Used Notebook,LAP1003,EcoBook Basic 15,2026-07-09,product,23000.0,Used,Laptop,2022,52000,14,0.6,1,0.5,55.769230769230774


 Create 
sku_component_metrics-Aggregates component-level information to the SKU level and 
calculates total component cost, weighted failure cost, and 
component reliability.

In [0]:
sku_component_metrics = (
    component_metrics
    .groupBy("sku")
    .agg(
        F.sum(F.col("component_cost")).alias("total_component_cost"),
        F.sum(
            F.col("component_cost") * F.col("failure_rate")
        ).alias("weighted_failure_cost")
    )
    .withColumn(
        "component_reliability_score",
        F.round(
            (
                1 -
                (
                    F.col("weighted_failure_cost")
                    / F.col("total_component_cost")
                )
            ) * 100,
            2
        )
    )
)

Join component metrics 
to gold_base-Adds the SKU-level component reliability information to the main 
Gold analytics dataset. 

In [0]:
gold_base = (
    gold_base
    .join(
        sku_component_metrics,
        on="sku",
        how="left"
    )
)

In [0]:
display(
    gold_base.select(
        "listing_id",
        "sku",
        "product_name",
        "secondary_market_price",
        "original_price",
        "secondary_market_depreciation",
        "total_component_cost",
        "weighted_failure_cost",
        "component_reliability_score"
    ).orderBy("listing_id")
)

listing_id,sku,product_name,secondary_market_price,original_price,secondary_market_depreciation,total_component_cost,weighted_failure_cost,component_reliability_score
M001,LAP1001,EcoBook Pro 14,42000.0,75000,44.0,45300.0,7110.15,84.3
M002,LAP1001,EcoBook Pro 14,38000.0,75000,49.333333333333336,45300.0,7110.15,84.3
M003,LAP1001,EcoBook Pro 14,9000.0,75000,88.0,45300.0,7110.15,84.3
M004,LAP1001,EcoBook Pro 14,4000.0,75000,94.66666666666667,45300.0,7110.15,84.3
M005,LAP1001,EcoBook Pro 14,2200.0,75000,97.06666666666666,45300.0,7110.15,84.3
M006,LAP1002,EcoBook Air 13,36000.0,68000,47.05882352941176,47565.0,7770.378,83.66
M007,LAP1002,EcoBook Air 13,32000.0,68000,52.94117647058824,47565.0,7770.378,83.66
M008,LAP1002,EcoBook Air 13,8200.0,68000,87.94117647058823,47565.0,7770.378,83.66
M009,LAP1003,EcoBook Basic 15,26000.0,52000,50.0,43035.0,7305.6804999999995,83.02
M010,LAP1003,EcoBook Basic 15,23000.0,52000,55.769230769230774,43035.0,7305.6804999999995,83.02


Calculate 
resale_retention_score-Calculates the percentage of the original product value that remains 
in the secondary market.

In [0]:
gold_base = gold_base.withColumn(
    "resale_retention_score",
    when(
        col("original_price") > 0,
        (
            col("secondary_market_price")
            / col("original_price")
        ) * 100
    ).otherwise(0.0)
)

In [0]:
display(
    gold_base.select(
        "listing_id",
        "sku",
        "original_price",
        "secondary_market_price",
        "secondary_market_depreciation",
        "resale_retention_score"
    ).orderBy("listing_id")
)

listing_id,sku,original_price,secondary_market_price,secondary_market_depreciation,resale_retention_score
M001,LAP1001,75000,42000.0,44.0,56.00000000000001
M002,LAP1001,75000,38000.0,49.333333333333336,50.66666666666667
M003,LAP1001,75000,9000.0,88.0,12.0
M004,LAP1001,75000,4000.0,94.66666666666667,5.333333333333334
M005,LAP1001,75000,2200.0,97.06666666666666,2.933333333333333
M006,LAP1002,68000,36000.0,47.05882352941176,52.94117647058824
M007,LAP1002,68000,32000.0,52.94117647058824,47.05882352941176
M008,LAP1002,68000,8200.0,87.94117647058823,12.058823529411764
M009,LAP1003,52000,26000.0,50.0,50.0
M010,LAP1003,52000,23000.0,55.769230769230774,44.230769230769226


Calculate 
circularity_score-Creates the proposed Circularity Score by combining resale retention 
and component reliability.

In [0]:
gold_base = gold_base.withColumn(
    "circularity_score",
    round(
        (
            col("resale_retention_score") * 0.50
            +
            col("component_reliability_score") * 0.50
        ),
        2
    )
)

In [0]:
display(
    gold_base.select(
        "listing_id",
        "sku",
        "resale_retention_score",
        "component_reliability_score",
        "circularity_score"
    ).orderBy("listing_id")
)

listing_id,sku,resale_retention_score,component_reliability_score,circularity_score
M001,LAP1001,56.00000000000001,84.3,70.15
M002,LAP1001,50.66666666666667,84.3,67.48
M003,LAP1001,12.0,84.3,48.15
M004,LAP1001,5.333333333333334,84.3,44.82
M005,LAP1001,2.933333333333333,84.3,43.62
M006,LAP1002,52.94117647058824,83.66,68.3
M007,LAP1002,47.05882352941176,83.66,65.36
M008,LAP1002,12.058823529411764,83.66,47.86
M009,LAP1003,50.0,83.02,66.51
M010,LAP1003,44.230769230769226,83.02,63.63


Save and display the 
final Gold table

In [0]:
gold_base.write.mode("overwrite").saveAsTable(
    "workspace.default.gold_echochain_analytics"
)

In [0]:
display(
    spark.table("workspace.default.gold_echochain_analytics")
)

sku,listing_id,marketplace_title,product_name,listing_date,listing_type,secondary_market_price,condition,category,launch_year,original_price,distance,token_score,size_match,weighted_score,secondary_market_depreciation,total_component_cost,weighted_failure_cost,component_reliability_score,resale_retention_score,circularity_score
LAP1001,M001,EcoBook Pro 14 Laptop 16GB,EcoBook Pro 14,2026-07-01,product,42000.0,Good,Laptop,2024,75000,12,0.6,1,0.5,44.0,45300.0,7110.15,84.3,56.00000000000001,70.15
LAP1001,M002,EcoBook Pro14 i7 Used Laptop,EcoBook Pro 14,2026-07-03,product,38000.0,Used,Laptop,2024,75000,16,0.42857142857142855,1,0.41428571428571426,49.333333333333336,45300.0,7110.15,84.3,50.66666666666667,67.48
LAP1001,M003,Eco Book Pro 14 Screen Display,EcoBook Pro 14,2026-07-05,component,9000.0,Used,Laptop,2024,75000,16,0.2857142857142857,1,0.34285714285714286,88.0,45300.0,7110.15,84.3,12.0,48.15
LAP1001,M004,EcoBook Pro 14 Battery Original,EcoBook Pro 14,2026-07-06,component,4000.0,Used,Laptop,2024,75000,17,0.6,1,0.5,94.66666666666667,45300.0,7110.15,84.3,5.333333333333334,44.82
LAP1001,M005,EcoBook Pro 14 Motherboard,EcoBook Pro 14,2026-07-07,component,2200.0,For Parts,Laptop,2024,75000,12,0.75,1,0.575,97.06666666666666,45300.0,7110.15,84.3,2.933333333333333,43.62
LAP1002,M006,EcoBook Air 13 Laptop,EcoBook Air 13,2026-07-02,product,36000.0,Good,Laptop,2023,68000,7,0.75,1,0.575,47.05882352941176,47565.0,7770.378,83.66,52.94117647058824,68.3
LAP1002,M007,EcoBook Air13 13-inch Used,EcoBook Air 13,2026-07-04,product,32000.0,Fair,Laptop,2023,68000,13,0.6,1,0.5,52.94117647058824,47565.0,7770.378,83.66,47.05882352941176,65.36
LAP1002,M008,EcoBook Air 13 Display Panel,EcoBook Air 13,2026-07-08,component,8200.0,Used,Laptop,2023,68000,14,0.6,1,0.5,87.94117647058823,47565.0,7770.378,83.66,12.058823529411764,47.86
LAP1003,M009,EcoBook Basic 15 Laptop,EcoBook Basic 15,2026-07-02,product,26000.0,Good,Laptop,2022,52000,7,0.75,1,0.575,50.0,43035.0,7305.6804999999995,83.02,50.0,66.51
LAP1003,M010,EcoBook Basic15 Used Notebook,EcoBook Basic 15,2026-07-09,product,23000.0,Used,Laptop,2022,52000,14,0.6,1,0.5,55.769230769230774,43035.0,7305.6804999999995,83.02,44.230769230769226,63.63
